# 1. Define constants

In [17]:
import os
import re

# Define the relative path to the data directory
data_folder = "../data"
YEARS = []

# Iterate through all files in the data folder to find the years
for filename in os.listdir(data_folder):
    # Check if the file is a JPEG image
    if filename.endswith((".jpg", ".png")):
        # Use regular expression to find the first sequence of digits in the filename
        match = re.search(r"\d+", filename)
        
        # If a number is found, add it to the list
        if match:
            YEARS.append(int(match.group()))

# Sort the years to maintain chronological order
YEARS.sort()

if not YEARS:
    raise ValueError("No years found in jpg files")

In [15]:
print(YEARS)

[2018, 2019, 2021, 2022, 2023, 2024, 2025]


In [12]:
FESTIVAL_NAME = "edc"

# 2. Collect DJ names from poster

In [18]:
import cv2
import pytesseract
import pandas as pd
import os
import re
import numpy as np

# Dictionary to store the lineup for each year
festival_lineups = {}

# Create extract folder if it doesn't exist
extract_folder = "../extract"
os.makedirs(extract_folder, exist_ok=True)
# Create debug folder to check what Tesseract sees
debug_folder = "../debug_images"
os.makedirs(debug_folder, exist_ok=True)

print("Starting OCR extraction...")

for year in YEARS:
    # Construct the filename based on the year
    possible_filenames = [f"{year}_edc_lineup.jpg", f"{year}_edc_lineup.png"]
    file_path = None
    
    for fname in possible_filenames:
        temp_path = os.path.join(data_folder, fname)
        if os.path.exists(temp_path):
            file_path = temp_path
            break
    
    if file_path is None:
        print(f"Warning: File not found for year {year}")
        continue
        
    # Read the image
    img = cv2.imread(file_path)
    
    if img is None:
        print(f"Error reading image for year {year}")
        continue

    # --- Advanced Preprocessing ---
    # 1. Resize (Upscale)
    scale_percent = 300 
    width = int(img.shape[1] * scale_percent / 100)
    height = int(img.shape[0] * scale_percent / 100)
    img_resized = cv2.resize(img, (width, height), interpolation=cv2.INTER_CUBIC)

    # 2. Specific Color Masking in HSV
    # The previous mask was too broad and picked up the neon background.
    # We now specifically target WHITE and YELLOW text, ignoring Pink/Blue/Purple.
    hsv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
    
    # Mask 1: White text (Strict)
    # S: 0-50 (Very low saturation = white/grey)
    # V: 180-255 (High brightness)
    lower_white = np.array([0, 0, 180])
    upper_white = np.array([180, 50, 255])
    mask_white = cv2.inRange(hsv, lower_white, upper_white)
    
    # Mask 2: Yellow text (for headers/top artists)
    # H: 20-35 (Yellow hue)
    # S: 100-255 (Vibrant)
    # V: 150-255 (Bright)
    lower_yellow = np.array([20, 100, 150])
    upper_yellow = np.array([35, 255, 255])
    mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
    
    # Combine masks
    mask = cv2.bitwise_or(mask_white, mask_yellow)
    
    # 3. Morphological Cleaning
    # Remove small noise dots (from background texture)
    kernel_clean = np.ones((2,2), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_clean)
    
    # 4. Invert for Tesseract (Black text on White background)
    processed_img = cv2.bitwise_not(mask)
    
    # 5. Slight Dilation (to thicken text)
    # Since we inverted, we erode the white background to dilate the black text
    processed_img = cv2.erode(processed_img, kernel_clean, iterations=1)
    
    # Save debug image
    debug_path = os.path.join(debug_folder, f"{year}_debug.jpg")
    cv2.imwrite(debug_path, processed_img)

    # --- Extraction ---
    try:
        # --psm 6: Assume a single uniform block of text
        text = pytesseract.image_to_string(processed_img, config='--psm 6')
    except Exception as e:
        print(f"OCR failed for {year}: {e}")
        continue
    
    # --- Cleaning ---
    cleaned_names = []
    
    # Keywords to exclude
    exclude_keywords = ["EDC", "LAS VEGAS", "MAY", "18", "19", "20", "21", "22", "23",
                        "FRIDAY", "SATURDAY", "SUNDAY", "YOU", "HEADLINER", "TICKETS", "COM", 
                        "INSOMNIAC", "MOTOR SPEEDWAY", "PASQUALE", "ROTELLA", "MUSIC", "FESTIVAL"]
    
    for line in text.split('\n'):
        # Skip lines containing excluded keywords (case insensitive)
        if any(keyword in line.upper() for keyword in exclude_keywords):
            continue
            
        # Split on common separators
        # Added more separators like ' b2b ' (case insensitive handled later)
        parts = re.split(r"•|\||::|\s-\s|\s\.\s|\s\*\s|\s\+\s|\s[bB]2[bB]\s", line)
        
        for part in parts:
            name = part.strip()
            # Filter out empty lines, very short strings, and pure numbers
            if len(name) > 2 and not name.isdigit():
                # Remove common OCR noise characters
                name = name.strip(".,-_*~|[](){}<>!@#$%^&")
                # Remove "B2B" if it stuck to the name
                name = re.sub(r"\s?[bB]2[bB]\s?", "", name)
                
                if len(name) > 2:
                    cleaned_names.append(name)
    
    festival_lineups[year] = cleaned_names
    
    # Save to CSV
    csv_filename = f"{year}_edc_artists.csv"
    csv_path = os.path.join(extract_folder, csv_filename)
    
    df = pd.DataFrame(cleaned_names, columns=["Artist"])
    df.to_csv(csv_path, index=False)
    
    print(f"Year {year}: Extracted {len(cleaned_names)} artists. Saved to {csv_path}")

print("\nExtraction complete. Check 'debug_images' folder to see what Tesseract saw.")
# Display a sample from the first available year
if YEARS:
    first_year = YEARS[0]
    print(f"\nSample extracted text for {first_year}:")
    print(festival_lineups[first_year][:10])

Starting OCR extraction...
Year 2018: Extracted 88 artists. Saved to ../extract/2018_edc_artists.csv
Year 2019: Extracted 43 artists. Saved to ../extract/2019_edc_artists.csv
Year 2021: Extracted 48 artists. Saved to ../extract/2021_edc_artists.csv
Year 2022: Extracted 14 artists. Saved to ../extract/2022_edc_artists.csv
Year 2023: Extracted 71 artists. Saved to ../extract/2023_edc_artists.csv
Year 2024: Extracted 61 artists. Saved to ../extract/2024_edc_artists.csv
Year 2025: Extracted 30 artists. Saved to ../extract/2025_edc_artists.csv

Extraction complete. Check 'debug_images' folder to see what Tesseract saw.

Sample extracted text for 2018:
['—_-_ Oe , :', ' > 4 ‘. NY', ' f S Aly) ', 'VAUUUL ULES / = lf', ' “ip ', 'tae ae Ment Diets', 'a on', 'yo mA', 'Tos oP SUEGENTS BEROT SOVRVER fae', 'SOP TNR LUHOPYOK © MO DINO']
